# NeMo Data Designer – Synthetic Data Generation Example

This notebook demonstrates how to use **NeMo Data Designer** (NDD) stage to generate synthetic medical-notes data from a small seed dataset.

The pipeline:
1. Downloads a public symptom-to-diagnosis CSV and converts it to JSONL shards.
2. Define the configuration for the NDD stage.
3. Build and run a Curator pipeline, composing of three stages: `JsonlReader`, `DataDesignerStage`, `JsonlWriter`.

### Imports libraries

In [1]:
import time
from pathlib import Path

import data_designer.config as dd
import pandas as pd

from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.synthetic.nemo_data_designer.data_designer import DataDesignerStage
from nemo_curator.stages.text.io.reader.jsonl import JsonlReader
from nemo_curator.stages.text.io.writer.jsonl import JsonlWriter
from nemo_curator.utils.file_utils import get_all_file_paths_under

/raid/huvu/venvs/curator_env_fast/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-31 14:46:09,801	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


### Download and Prepare Seed Data

The seed dataset is a public symptom-to-diagnosis CSV hosted on GitHub.  
We download it, split it into small JSONL shards (10 rows each), and save them locally so the pipeline can read them in parallel.
We sample 200 records from the original dataset for testing purposes.

In [2]:
SEED_CSV_URL = (
    "https://raw.githubusercontent.com/NVIDIA/GenerativeAIExamples/refs/heads/main"
    "/nemo/NeMo-Data-Designer/data/gretelai_symptom_to_diagnosis.csv"
)


def download_and_convert_seed_data(
    output_dir: str | Path | None = None,
    records_per_file: int = 10,
    number_of_records: int = 200,
) -> str:
    """Download seed CSV from URL, convert to JSONL (chunked), return output dir path."""
    if output_dir is None:
        output_dir = Path("processed_seed_data")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(SEED_CSV_URL, sep=",", encoding="utf-8")
    df = df.head(number_of_records)
    for i, start in enumerate(range(0, len(df), records_per_file)):
        chunk = df.iloc[start : start + records_per_file]
        chunk.to_json(
            output_dir / f"{i:06d}.jsonl",
            orient="records",
            lines=True,
            force_ascii=False,
            date_format="iso",
        )
    return str(output_dir)


print("Preparing seed data (download + CSV→JSONL)...")
seed_dir = download_and_convert_seed_data()
print(f"Seed data ready: {seed_dir}")

Preparing seed data (download + CSV→JSONL)...
Seed data ready: processed_seed_data


### Configuration

Set the output directory for generated data here.  
You can also point `DATA_DESIGNER_CONFIG_FILE` at a YAML config file; leave it as `None` to use the programmatic config defined in this notebook.

In [3]:
OUTPUT_PATH = "./synthetic_output"
DATA_DESIGNER_CONFIG_FILE = None  # set to a file path string to load a YAML config

### Define the Data Designer Config

The DataDesignerStage is intialized from a NDD's configuration builder (https://nvidia-nemo.github.io/DataDesigner/latest/code_reference/config_builder/). In this step, we manually create a configuration for the NDD module. This includes setting the LLM model provider, and the steps taken inside NDD module. Users can learn more about NDD's API from NDD official documentation: https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/.

In this tutorial, we replicate the official NDD's tutorials on synthetic data genertion with seed data (https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/), but with Nemo-Curator.


In [4]:
def _build_config(model_id: str, provider_name: str, model_alias: str) -> dd.DataDesignerConfigBuilder:
    """Build the Data Designer config with medical notes generation."""
    model_configs = [
        dd.ModelConfig(
            alias=model_alias,
            model=model_id,
            provider=provider_name,
            skip_health_check=True,
            inference_parameters=dd.ChatCompletionInferenceParams(
                temperature=1.0,
                top_p=1.0,
                max_tokens=2048,
            ),
        )
    ]

    config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="patient_sampler",
            sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
            params=dd.PersonFromFakerSamplerParams(),
        )
    )

    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="doctor_sampler",
            sampler_type=dd.SamplerType.PERSON_FROM_FAKER,
            params=dd.PersonFromFakerSamplerParams(),
        )
    )

    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="patient_id",
            sampler_type=dd.SamplerType.UUID,
            params=dd.UUIDSamplerParams(
                prefix="PT-",
                short_form=True,
                uppercase=True,
            ),
        )
    )

    config_builder.add_column(
        dd.ExpressionColumnConfig(
            name="first_name",
            expr="{{ patient_sampler.first_name}}",
        )
    )

    config_builder.add_column(
        dd.ExpressionColumnConfig(
            name="last_name",
            expr="{{ patient_sampler.last_name }}",
        )
    )

    config_builder.add_column(
        dd.ExpressionColumnConfig(
            name="dob",
            expr="{{ patient_sampler.birth_date }}",
        )
    )

    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="symptom_onset_date",
            sampler_type=dd.SamplerType.DATETIME,
            params=dd.DatetimeSamplerParams(start="2024-01-01", end="2024-12-31"),
        )
    )

    config_builder.add_column(
        dd.SamplerColumnConfig(
            name="date_of_visit",
            sampler_type=dd.SamplerType.TIMEDELTA,
            params=dd.TimeDeltaSamplerParams(
                dt_min=1,
                dt_max=30,
                reference_column_name="symptom_onset_date",
            ),
        )
    )

    config_builder.add_column(
        dd.ExpressionColumnConfig(
            name="physician",
            expr="Dr. {{ doctor_sampler.last_name }}",
        )
    )

    config_builder.add_column(
        dd.LLMTextColumnConfig(
            name="physician_notes",
            prompt="""\
    You are a primary-care physician who just had an appointment with {{ first_name }} {{ last_name }},
    who has been struggling with symptoms from {{ diagnosis }} since {{ symptom_onset_date }}.
    The date of today's visit is {{ date_of_visit }}.

    {{ patient_summary }}

    Write careful notes about your visit with {{ first_name }},
    as Dr. {{ doctor_sampler.first_name }} {{ doctor_sampler.last_name }}.

    Format the notes as a busy doctor might.
    Respond with only the notes, no other text.
    """,
            model_alias=model_alias,
        )
    )

    return config_builder

### Start Ray client

User can add `NVIDIA_API_KEY` by running `os.environ["NVIDIA_API_KEY"] = "YOUR_API_KEY"` if needed.

In [ ]:
client = RayClient()  # change as needed
client.start()

2026-03-31 14:46:53.670 | INFO     | nemo_curator.core.utils:init_cluster:185 - Ray start command: ray start --head --node-ip-address 127.0.1.1 --port 6381 --metrics-export-port 8082 --dashboard-host 127.0.0.1 --dashboard-port 8267 --ray-client-server-port 20002 --temp-dir /tmp/ray_huvu --disable-usage-stats --num-gpus 1 --num-cpus 8 --block


2026-03-31 14:46:54,830	INFO usage_lib.py:447 -- Usage stats collection is disabled.
2026-03-31 14:46:54,830	INFO scripts.py:936 -- Local node IP: 127.0.1.1
2026-03-31 14:47:09,078	SUCC scripts.py:975 -- --------------------
2026-03-31 14:47:09,079	SUCC scripts.py:976 -- Ray runtime started.
2026-03-31 14:47:09,079	SUCC scripts.py:977 -- --------------------
2026-03-31 14:47:09,079	INFO scripts.py:979 -- Next steps
2026-03-31 14:47:09,079	INFO scripts.py:982 -- To add another node to this Ray cluster, run
2026-03-31 14:47:09,079	INFO scripts.py:985 --   ray start --address='127.0.1.1:6381'
2026-03-31 14:47:09,079	INFO scripts.py:996 -- To connect to this Ray cluster:
2026-03-31 14:47:09,079	INFO scripts.py:998 -- import ray
2026-03-31 14:47:09,079	INFO scripts.py:999 -- ray.init(_node_ip_address='127.0.1.1')
2026-03-31 14:47:09,079	INFO scripts.py:1013 -- To submit a Ray job using the Ray Jobs CLI:
2026-03-31 14:47:09,079	INFO scripts.py:1014 --   RAY_API_SERVER_ADDRESS='http://127.0.0

### Build the NeMo Curator Pipeline

After defining the NDD configuration which will be used to initialize `DataDesignerStage` stage, we build the pipeline chains three stages:

| # | Stage | Role |
|---|-------|------|
| 1 | `JsonlReader` | Read seed JSONL shards from disk |
| 2 | `DataDesignerStage` | Enrich each record using the NDD config |
| 3 | `JsonlWriter` | Write enriched records back to JSONL |



In [ ]:
provider = None # when set to None we use local Ray inference, set to "nvidia" to use Nvidia endpoint
model_alias = "local-llm"
provider_name = provider or "local"
inference_server = None
model_providers = None
model= "openai/gpt-oss-20b" # when use Nvidia endpoint, set to "meta/llama-3.3-70b-instruct" as an example

If provider is None, launch Ray inference through vLLM, which requires GPUs locally. Otherwise, using the default Nvidia's endpoint.

In [15]:
# If no remote provider specified, start a local InferenceServer
if provider is None:
    from nemo_curator.backends.experimental.utils import get_available_cpu_gpu_resources
    from nemo_curator.core.serve import InferenceModelConfig, InferenceServer

    _, num_gpus = get_available_cpu_gpu_resources(init_and_shutdown=True)
    num_gpus = int(num_gpus)
    print(f"Detected {num_gpus} GPUs, using tensor_parallel_size={num_gpus}")

    server_config = InferenceModelConfig(
        model_identifier=model,
        deployment_config={
            "autoscaling_config": {
                "min_replicas": 1,
                "max_replicas": 1,
            },
        },
        engine_kwargs={
            "tensor_parallel_size": num_gpus,
        },
    )

    inference_server = InferenceServer(models=[server_config])
    inference_server.start()

    # Create a custom ModelProvider pointing at the local server
    model_providers = [
        dd.ModelProvider(
            name=provider_name,
            endpoint=inference_server.endpoint,
            api_key="unused",  # pragma: allowlist secret
        )
    ]
    print(f"Local InferenceServer ready at {inference_server.endpoint}")

2026-03-31 14:51:26,294	INFO worker.py:1669 -- Using address 127.0.1.1:6381 set in the environment variable RAY_ADDRESS
2026-03-31 14:51:26,302	INFO worker.py:1810 -- Connecting to existing Ray cluster at address: 127.0.1.1:6381...
2026-03-31 14:51:26,435	INFO worker.py:2004 -- Connected to Ray cluster. View the dashboard at 127.0.0.1:8267 


Detected 1 GPUs, using tensor_parallel_size=1


2026-03-31 14:51:27,211	INFO worker.py:1669 -- Using address 127.0.1.1:6381 set in the environment variable RAY_ADDRESS
2026-03-31 14:51:27,219	INFO worker.py:1810 -- Connecting to existing Ray cluster at address: 127.0.1.1:6381...
2026-03-31 14:51:27,342	INFO worker.py:2004 -- Connected to Ray cluster. View the dashboard at 127.0.0.1:8267 
2026-03-31 14:51:27.361 | INFO     | nemo_curator.core.serve:_deploy:227 - Starting Ray Serve with models: ['openai/gpt-oss-20b'] on port 8000
INFO 2026-03-31 14:51:36,588 serve 1369491 -- ============== Deployment Options ==============
INFO 2026-03-31 14:51:36,590 serve 1369491 -- {'autoscaling_config': {'max_replicas': 1,
                        'min_replicas': 1,
                        'target_ongoing_requests': 1000000000},
 'health_check_period_s': 10,
 'health_check_timeout_s': 10,
 'max_ongoing_requests': 1000000000,
 'name': 'LLMServer:openai--gpt-oss-20b',
 'placement_group_bundles': [{'CPU': 1, 'GPU': 1}],
 'placement_group_strategy': 'P

(_get_vllm_engine_config pid=1402166) WARNING 03-31 14:51:57 [vllm.py:613] Async scheduling will be disabled because it is not supported with the `ray` distributed executor backend (only `mp`, `uni`, and `external_launcher` are supported).


(ServeReplica:default:LLMServer:openai--gpt-oss-20b pid=1401871) You are using a model of type gpt_oss to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
(ServeReplica:default:LLMServer:openai--gpt-oss-20b pid=1401871) You are using a model of type gpt_oss to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


...

(RayWorkerWrapper pid=1403177) WARNING 03-31 14:52:37 [marlin_utils_fp4.py:336] Your GPU does not have native support for FP4 computation but FP4 quantization is being used. Weight-only FP4 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.
(ServeReplica:default:LLMServer:openai--gpt-oss-20b pid=1401871) (EngineCore_DP0 pid=1402804) (RayWorkerWrapper pid=1403177) WARNING 03-31 14:52:17 [system_utils.py:37] Overwriting environment variable LD_LIBRARY_PATH from '/raid/huvu/venvs/curator_env_fast/lib/python3.10/site-packages/cv2/../../lib64:' to '/raid/huvu/venvs/curator_env_fast/lib/python3.10/site-packages/cv2/../../lib64:/raid/huvu/venvs/curator_env_fast/lib/python3.10/site-packages/cv2/../../lib64:'
(ServeReplica:default:LLMServer:openai--gpt-oss-20b pid=1401871) (EngineCore_DP0 pid=1402804) (RayWorkerWrapper pid=1403177) WARNING 03-31 14:52:19 [worker_base.py:301] Missing `shared_worker_lock` argument from executor. This a

(ServeController pid=1381120) WARNING 2026-03-31 14:52:38,803 controller 1381120 -- Deployment 'LLMServer:openai--gpt-oss-20b' in application 'default' has 1 replicas that have taken more than 30s to initialize.
(ServeController pid=1381120) This may be caused by a slow __init__ or reconfigure method.
(ServeController pid=1381120) WARNING 2026-03-31 14:52:38,803 controller 1381120 -- Deployment 'OpenAiIngress' in application 'default' has 1 replicas that have taken more than 30s to initialize.
(ServeController pid=1381120) This may be caused by a slow __init__ or reconfigure method.
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/83 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:13<00:00,  4.44s/it] [repeated 2x across cluster]RayWorkerWrapper pid=1403177) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   1%|          | 1/83 [00:00<00:08,  9.96it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  93%|████

(ServeReplica:default:LLMServer:openai--gpt-oss-20b pid=1401871) WARNING 03-31 14:53:51 [serving.py:242] For gpt-oss, we ignore --enable-auto-tool-choice and always enable tool use.
(ServeReplica:default:LLMServer:openai--gpt-oss-20b pid=1401871) (EngineCore_DP0 pid=1402804) (RayWorkerWrapper pid=1403177) WARNING 03-31 14:52:37 [marlin_utils_fp4.py:336] Your GPU does not have native support for FP4 computation but FP4 quantization is being used. Weight-only FP4 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.


INFO 2026-03-31 14:53:59,093 serve 1369491 -- Application 'default' is ready at http://127.0.0.1:8000/.
2026-03-31 14:53:59.286 | INFO     | nemo_curator.core.serve:_wait_for_healthy:384 - Model server ready after 1 health check(s)
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:51<00:00,  1.47s/it]ineCore_DP0 pid=1402804) (RayWorkerWrapper pid=1403177) 
2026-03-31 14:53:59.547 | INFO     | nemo_curator.core.serve:start:217 - Ray Serve is ready at http://localhost:8000/v1


Local InferenceServer ready at http://localhost:8000/v1


Setup data designer configuration.

In [16]:
if DATA_DESIGNER_CONFIG_FILE is not None:
    config_builder = dd.DataDesignerConfigBuilder.from_config(DATA_DESIGNER_CONFIG_FILE)
else:
    config_builder = _build_config(
        model_id=model,
        provider_name=provider_name,
        model_alias=model_alias,
    )

Create running pipeline.

**Note:** Here for illustration purpose, we set `num_workers` to low values to avoid exceeding low rate limit. Users can omit this step if rate limit is not a concern.

In [17]:
pipeline = Pipeline(
    name="ndd_data_generation",
    description="Generate synthetic text data using Nemo Data Designer",
)

pipeline.add_stage(
    JsonlReader(
        file_paths=seed_dir + "/*.jsonl",
        fields=["diagnosis", "patient_summary"],
    )
)

# Add the Nemo Data Designer stage (limit actors to stay within API rate limits)
ndd_stage = DataDesignerStage(config_builder=config_builder, model_providers=model_providers)
ndd_stage.xenna_stage_spec = lambda: {"num_workers": 2}
pipeline.add_stage(ndd_stage)

pipeline.add_stage(JsonlWriter(path=OUTPUT_PATH))

print(pipeline.describe())

2026-03-31 14:55:25.409 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'jsonl_reader' to pipeline 'ndd_data_generation'
2026-03-31 14:55:25.411 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'DataDesignerStage' to pipeline 'ndd_data_generation'
2026-03-31 14:55:25.443 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'jsonl_writer' to pipeline 'ndd_data_generation'


Pipeline: ndd_data_generation
Description: Generate synthetic text data using Nemo Data Designer
Stages: 3

Stage 1: jsonl_reader
  Resources: 1.0 CPUs
  Batch size: 1
  Outputs:
    Output attributes: data
    Output columns: diagnosis, patient_summary
Stage 2: DataDesignerStage
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required attributes: data
  Outputs:
    Output attributes: data
Stage 3: jsonl_writer
  Resources: 1.0 CPUs
  Batch size: 1
  Inputs:
    Required attributes: data
  Outputs:
    Output attributes: data



### Run the Pipeline

In [18]:
print("Starting synthetic data generation pipeline...")
start_time = time.time()
pipeline.run()
end_time = time.time()

elapsed_time = end_time - start_time
print("\nPipeline completed!")
print(f"Total execution time: {elapsed_time:.2f} seconds ({elapsed_time / 60:.2f} minutes)")

2026-03-31 14:55:28.401 | INFO     | nemo_curator.pipeline.pipeline:build:70 - Planning pipeline: ndd_data_generation
2026-03-31 14:55:28.402 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:106 - Decomposing composite stage: jsonl_reader
2026-03-31 14:55:28.402 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:120 - Expanded 'jsonl_reader' into 2 execution stages
2026-03-31 14:55:28.403 | INFO     | nemo_curator.backends.xenna.executor:execute:135 - Execution mode: STREAMING
2026-03-31 14:55:28,404	INFO worker.py:1669 -- Using address 127.0.1.1:6381 set in the environment variable RAY_ADDRESS
2026-03-31 14:55:28,411	INFO worker.py:1810 -- Connecting to existing Ray cluster at address: 127.0.1.1:6381...
2026-03-31 14:55:28,548	INFO worker.py:2004 -- Connected to Ray cluster. View the dashboard at 127.0.0.1:8267 
2026-03-31 14:55:28.573 | INFO     | nemo_curator.backends.xenna.adapter:required_resources:43 - Resources: Resources(cpus=0.5, gpu_memory_gb=0.0, gpus

Starting synthetic data generation pipeline...


2026-03-31 14:55:29.564 | INFO     | nemo_curator.backends.xenna.adapter:required_resources:43 - Resources: Resources(cpus=0.5, gpu_memory_gb=0.0, gpus=0.0)
2026-03-31 14:55:29.565 | INFO     | nemo_curator.backends.xenna.adapter:required_resources:43 - Resources: Resources(cpus=1.0, gpu_memory_gb=0.0, gpus=0.0)
2026-03-31 14:55:29.565 | INFO     | nemo_curator.backends.xenna.adapter:required_resources:43 - Resources: Resources(cpus=1.0, gpu_memory_gb=0.0, gpus=0.0)
2026-03-31 14:55:29.566 | INFO     | nemo_curator.backends.xenna.adapter:required_resources:43 - Resources: Resources(cpus=1.0, gpu_memory_gb=0.0, gpus=0.0)
2026-03-31 14:55:29.566 | INFO     | cosmos_xenna.pipelines.private.pipelines:run_pipeline:484 - Cluster resources: ClusterResources(nodes={'8cb2464e86bb476e004fe7c5364a1c29818ca7e58df1e1ec1d3afd6a': NodeResources(used_cpus=0.0, total_cpus=7, gpus=[GpuResources(index=1, uuid_=UUID('dd398c2c-c263-6d18-cfd6-1704e8b6dbb8'), used_fraction=0.0)], name='8cb2464e86bb476e004fe7


Pipeline completed!
Total execution time: 352.16 seconds (5.87 minutes)


### Inspect the Output

We can find the output results stored in `./synthetic_output` directory.

In [19]:
output_files = get_all_file_paths_under(OUTPUT_PATH, recurse_subdirectories=True, keep_extensions=".jsonl")

print(f"Generated data saved to: {OUTPUT_PATH}")
for file_path in output_files:
    print(f"  - {Path(file_path).name}")

all_data_frames = [pd.read_json(f, lines=True) for f in output_files]

Generated data saved to: ./synthetic_output
  - 2a32c234df7d.jsonl
  - 383a455f5a8d.jsonl
  - 44767c13dcd9.jsonl
  - 572222cbb959.jsonl
  - 5ed1794c3d4a.jsonl
  - 642cda8b7e5b.jsonl
  - 6af70db0176e.jsonl
  - 6d23d6c30b3e.jsonl
  - 6eedcc8ded3c.jsonl
  - 7080f45e14c9.jsonl
  - 7b44d21ee99f.jsonl
  - 7d220206227d.jsonl
  - 7ec98cc83a2c.jsonl
  - 884f8629a7fd.jsonl
  - 8928c29fad4e.jsonl
  - c7fde8d4971e.jsonl
  - cc45e91592c4.jsonl
  - ead3a316b626.jsonl
  - f7f60d292b3e.jsonl
  - fb1bb122478d.jsonl


In [20]:
if all_data_frames:
    combined_df = pd.concat(all_data_frames, ignore_index=True)
    print(f"Total records generated: {len(combined_df)}")

Total records generated: 200


Inspecting output .jsonl files content, we can find the generated text from Nvidia's LLM endpoint corresponding to the input prompt.

In [21]:
print("=" * 50)
print("Sample of generated documents:")
print("=" * 50)

for i, df in enumerate(all_data_frames[:3]):
    print(f"\nFile {i + 1}: {Path(output_files[i]).name}")
    print(f"Number of documents: {len(df)}")
    print("\nGenerated text (showing first record):")
    for j, row in enumerate(df.head(1).to_dict(orient="records")):
        print(f"Document {j + 1}:")
        for key, value in row.items():
            print(f"[{key}]:")
            print(f"{value}")
        print("-" * 40)

Sample of generated documents:

File 1: 2a32c234df7d.jsonl
Number of documents: 10

Generated text (showing first record):
Document 1:
[diagnosis]:
malaria
[patient_summary]:
I have a high fever, chills, and severe itching. I also have a headache, excessive sweating, nausea, and muscle aches.
[patient_sampler]:
{'uuid': '13a36ba1-5f8f-431c-86d1-372975b6c220', 'locale': 'en_US', 'first_name': 'Jeffrey', 'last_name': 'Watts', 'middle_name': None, 'sex': 'Male', 'street_number': '293', 'street_name': 'Harris Oval', 'city': 'New Jessica', 'state': 'Texas', 'postcode': '20375', 'age': 49, 'birth_date': '1976-05-09', 'country': 'American Samoa', 'marital_status': 'separated', 'education_level': 'secondary_education', 'unit': '', 'occupation': 'Educational psychologist', 'phone_number': '(484)575-5193', 'bachelors_field': 'no_degree'}
[doctor_sampler]:
{'uuid': '8b79d082-0a4e-47b0-b890-ba69ee9c2c6d', 'locale': 'en_US', 'first_name': 'Rachael', 'last_name': 'Cortez', 'middle_name': None, 'sex'